# CLIP

在分类任务中，一个一般的流程是：将不同类别用一个数字标签表示。例如：

$$
[0, 1, 2, 3, 4]
$$

可以表示一个五分类任务。深度学习模型学习的是输入到类别的映射，或者直接输出类别标签，或者更常见地输出每个类别的得分 logits：

$$
logits = f(x) \in \mathbb{R}^{N}
$$

其中 $N$ 是类别数量。之后再通过 softmax 得到每个类别的概率分布。

这种方式存在一些缺陷。当类别数量确定后，模型的输出维度也就固定了。如果训练时只训练了 5 分类，测试时多出几个新类别，例如 6、7、8，模型就很难直接泛化过去。因为分类标签本身是人为定义的数字索引，并没有自然语义。模型可以在训练数据中学习到“某类图片对应某个 index”，但这个 index 本身并不能告诉模型新类别是什么意思。

也就是说，传统分类头的问题是：

> 类别被压缩成了一个人为编号，模型学到的是 index 的对应关系，而不是类别本身的语义关系。

## CLIP 的核心思想

对比学习（CLIP）是一种目前已经得到广泛应用的训练方式，主要用于预训练 Encoder，让文本和其他模态对齐，也经常用于 open vocabulary classification，也就是开放词表分类任务。

CLIP 的思想是：不再直接把图像映射成固定类别 id，而是把图像和文本都映射到同一个 embedding 空间中，然后通过相似度判断它们是否匹配。

以图像-文本对齐为例，假设一个 batch 中有 $n$ 张图片和 $n$ 条对应文本。图像 Encoder 得到图像特征：

$$
I \in \mathbb{R}^{n \times C}
$$

文本 Encoder 得到文本特征：

$$
T \in \mathbb{R}^{n \times C}
$$

通常会先对 embedding 做归一化：

$$
\hat{I}_i = \frac{I_i}{\|I_i\|_2}, \quad \hat{T}_j = \frac{T_j}{\|T_j\|_2}
$$

这样点积就等价于余弦相似度。然后计算图像和文本之间的相似度矩阵：

$$
S = \hat{I}\hat{T}^T
$$

其中：

$$
S \in \mathbb{R}^{n \times n}
$$

$S_{i,j}$ 表示第 $i$ 张图片和第 $j$ 条文本之间的相似度。如果 batch 中的数据是一一配对的，那么对角线位置就是正样本：

$$
(i, i) \quad i = 1,2,\cdots,n
$$

非对角线位置则可以看作当前 batch 内的负样本。

![CLIP contrastive learning](../figs/clip.png)

## CLIP 的 loss

CLIP 使用对称的 cross entropy loss。

首先引入一个温度系数 $\tau$，用来控制 logits 的尺度。

$$
logits = \frac{\hat{I}\hat{T}^T}{\tau}
$$

代码中使用的是可学习的 `logit_scale`：

$$
logits = \exp(logit\_scale) \cdot \hat{I}\hat{T}^T
$$

当 scale 较小时，logits 之间的差异会被放大，使 softmax 输出分布更加尖锐：原本较大的 logit 会获得更高的概率，而较小的 logit 会被进一步压低；当 scale 较大时，logits 之间的差异被缩小，softmax 输出则更加平滑。

因此，温度系数的作用本质上是控制模型对不同 logits 差异的敏感程度，也就是控制 softmax 分布的“尖锐程度”。

对于图像到文本方向，每张图片都要在 $n$ 条文本中找到与自己匹配的那一条，因此 target 是：

$$
labels = [0, 1, 2, \cdots, n-1]
$$

图像到文本的 loss 为：

$$
L_{image} = CE(logits_{image}, labels)
$$

文本到图像方向同理，每条文本也要在 $n$ 张图片中找到与自己匹配的那一张：

$$
L_{text} = CE(logits_{text}, labels)
$$

最终 CLIP loss 是两者的平均：

$$
L = \frac{L_{image} + L_{text}}{2}
$$

这样训练之后，匹配的图文 embedding 会被拉近，不匹配的图文 embedding 会被推远。

## 用于 open vocabulary 分类

CLIP 用于分类任务时，不再需要重新训练一个固定类别数的分类头，而是可以把类别名称写成自然语言文本。

例如候选类别是：

$$
[\text{cat}, \text{dog}, \text{car}, \text{plane}]
$$

我们可以把它们写成 prompt：

$$
\text{a photo of a cat}
$$

$$
\text{a photo of a dog}
$$

$$
\text{a photo of a car}
$$

然后用 text encoder 得到候选文本 embedding。对于一张输入图片，先经过 image encoder 得到图像 embedding，再和所有候选文本 embedding 计算相似度：

$$
score_j = \hat{I}\hat{T}_j^T
$$

最后选择相似度最大的文本标签作为分类结果：

$$
\hat{y} = \arg\max_j score_j
$$

这里需要注意，是选择相似度最大的标签，而不是相似度最小的标签。

这种方式的优势是类别空间来自自然语言。只要能写出类别描述，就可以把它放进候选文本里，让模型进行 zero-shot prediction。因此 CLIP 被广泛用于多模态对齐、开放词表分类、图文检索和视觉语言模型的预训练中。

总的来说，CLIP 通过优化图像和文本 embedding 的相似度，让不同模态进入同一个语义空间。它不再只学习“图片 -> 人为类别 index”的映射，而是学习“图片语义 -> 文本语义”的匹配关系，因此泛化能力更强，也更适合开放词表场景。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from transformers import AutoModel, AutoTokenizer


class CLIP(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()

        # -----------------------
        # Image Encoder
        # -----------------------
        resnet = models.resnet50(weights="DEFAULT")

        image_dim = resnet.fc.in_features  # 2048
        resnet.fc = nn.Identity()

        self.image_encoder = resnet

        # -----------------------
        # Text Encoder
        # -----------------------
        self.text_encoder = AutoModel.from_pretrained(
            "bert-base-uncased"
        )

        text_dim = 768

        # -----------------------
        # Projection heads
        # -----------------------
        self.image_projection = nn.Linear(
            image_dim,
            embed_dim
        )

        self.text_projection = nn.Linear(
            text_dim,
            embed_dim
        )

        # CLIP temperature
        self.logit_scale = nn.Parameter(
            torch.tensor(1 / 0.07).log()
        )

    def encode_image(self, image):
        x = self.image_encoder(image)
        # [B, 2048]

        x = self.image_projection(x)
        # [B, 512]

        x = F.normalize(x, dim=-1)

        return x

    def encode_text(self, input_ids, attention_mask):
        outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # 简化：直接取 CLS
        x = outputs.last_hidden_state[:, 0]
        # [B, 768]

        x = self.text_projection(x)
        # [B, 512]

        x = F.normalize(x, dim=-1)

        return x

    def forward(
        self,
        image,
        input_ids,
        attention_mask
    ):
        image_features = self.encode_image(image)
        text_features = self.encode_text(
            input_ids,
            attention_mask
        )

        scale = self.logit_scale.exp()

        logits_per_image = (
            scale
            * image_features
            @ text_features.T
        )
        # [B, B]

        logits_per_text = logits_per_image.T
        # [B, B]

        return logits_per_image, logits_per_text

    def loss(
        self,
        logits_per_image,
        logits_per_text
    ):
        batch_size = logits_per_image.shape[0]

        labels = torch.arange(
            batch_size,
            device=logits_per_image.device
        ) # 每一行正确的标签位置

        image_loss = F.cross_entropy(
            logits_per_image,
            labels
        )

        text_loss = F.cross_entropy(
            logits_per_text,
            labels
        )

        return (image_loss + text_loss) / 2

In [ ]:
# Example usage
# 这里需要下载 ResNet50 和 BERT 的预训练权重；如果本地没有缓存，需要联网运行。
texts = ["An image of a cat"]
images = torch.rand(1, 3, 224, 224)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = tokenizer(texts, padding=True, return_tensors="pt")

model = CLIP()
logits_per_image, logits_per_text = model(
    images,
    tokens["input_ids"],
    tokens["attention_mask"],
)

print("logits_per_image shape:", logits_per_image.shape)
print("logits_per_text shape:", logits_per_text.shape)
